## ABSA K-Fold BERT Pipeline

In [1]:
import copy
import os
import random
from ast import literal_eval

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as utils
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold
from transformers import BertForSequenceClassification, BertModel, BertTokenizer

SEED = 1345
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

FC_HIDDEN = 256
LSTM_HIDDEN = 128
BERT_HIDDEN = 768
BERT_LAYERS = 12

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

d:\System\Conda\envs\absa\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


## 1. Data Utilities

In [2]:
def load_semeval(dataset, num_classes, data_path='../Data/'):
    label_map = {'negative': 0, 'positive': 1, 'neutral': 2, 'conflict': 3}

    train_file = f"{data_path}atsa-{dataset}/atsa_train.json"
    test_file = f"{data_path}atsa-{dataset}/atsa_test.json"

    train_raw = literal_eval(open(train_file, 'r', encoding='latin1').read())
    test_raw = literal_eval(open(test_file, 'r', encoding='latin1').read())

    def unpack(rows):
        sentences, aspects, sentiments = [], [], []
        for row in rows:
            if row['sentiment'] == 'conflict' and num_classes == 3:
                continue
            sentences.append(row['sentence'])
            aspects.append(row['aspect'])
            sentiments.append(label_map[row['sentiment']])
        return sentences, aspects, sentiments

    train_sentence, train_aspect, train_sentiment = unpack(train_raw)
    test_sentence, test_aspect, test_sentiment = unpack(test_raw)

    print('----------------------------------------')
    print(f"Max train sentence length: {max(len(s.split()) for s in train_sentence)}")
    print(f"Max train aspect length: {max(len(a.split()) for a in train_aspect)}")
    print('----------------------------------------')

    return (
        train_sentence,
        train_aspect,
        train_sentiment,
        test_sentence,
        test_aspect,
        test_sentiment,
    )

In [3]:
def get_loader(input_ids, attention_masks, token_type_ids, labels, batchsize, shuffle=False):
    """Converts tensors into a dataloader."""
    array = utils.TensorDataset(input_ids, attention_masks, token_type_ids, labels)
    loader = utils.DataLoader(
        array,
        batch_size=batchsize,
        shuffle=shuffle,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )
    return loader

## 2. Tokenization

In [4]:
def get_pretrained_tokenizer():
    return BertTokenizer.from_pretrained('bert-base-uncased', do_lower_case=True)


def tokenize_sentences(tokenizer, sentences, aspects, max_len):
    input_ids, attention_masks, token_type_ids = [], [], []

    for sentence, aspect in zip(sentences, aspects):
        encoded = tokenizer(
            text=sentence,
            text_pair=aspect,
            add_special_tokens=True,
            max_length=max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_token_type_ids=True,
            return_tensors='pt',
        )

        input_ids.append(encoded['input_ids'])
        attention_masks.append(encoded['attention_mask'])
        token_type_ids.append(encoded['token_type_ids'])

    return (
        torch.cat(input_ids, dim=0),
        torch.cat(attention_masks, dim=0),
        torch.cat(token_type_ids, dim=0),
    )

## 3. Model Definitions

In [5]:
class Bert_Base(nn.Module):
    def __init__(self, numclasses):
        super().__init__()
        self.bert = BertForSequenceClassification.from_pretrained(
            'bert-base-uncased',
            output_hidden_states=False,
            output_attentions=False,
            num_labels=numclasses,
        )

    def forward(self, inp_ids, att_mask, token_ids, labels):
        outputs = self.bert(
            input_ids=inp_ids,
            attention_mask=att_mask,
            token_type_ids=token_ids,
            labels=labels,
        )
        return outputs.loss, outputs.logits

In [6]:
class Bert_LSTM(nn.Module):
    def __init__(self, numclasses):
        super().__init__()
        self.numlayers = BERT_LAYERS
        self.embeddim = BERT_HIDDEN

        self.bert = BertModel.from_pretrained(
            'bert-base-uncased',
            output_hidden_states=True,
            output_attentions=False,
        )
        self.lstm = nn.LSTM(self.embeddim, LSTM_HIDDEN, batch_first=True)
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(LSTM_HIDDEN, numclasses)

    def forward(self, inp_ids, att_mask, token_ids):
        outputs = self.bert(
            input_ids=inp_ids,
            attention_mask=att_mask,
            token_type_ids=token_ids,
        )
        hidden_states = outputs.hidden_states

        cls_stack = torch.stack(
            [hidden_states[layer_i][:, 0].squeeze() for layer_i in range(0, self.numlayers)],
            dim=-1,
        )
        cls_stack = cls_stack.view(-1, self.numlayers, self.embeddim)

        out, _ = self.lstm(cls_stack, None)
        out = self.dropout(out[:, -1, :])
        return self.fc(out)

In [7]:
class Bert_Attention(nn.Module):
    def __init__(self, numclasses):
        super().__init__()
        self.numlayers = BERT_LAYERS
        self.embeddim = BERT_HIDDEN

        self.bert = BertModel.from_pretrained(
            'bert-base-uncased',
            output_hidden_states=True,
            output_attentions=False,
        )
        self.q = nn.Parameter(torch.from_numpy(np.random.normal(0.0, 0.1, (1, self.embeddim))).float())
        self.w_h = nn.Parameter(torch.from_numpy(np.random.normal(0.0, 0.1, (self.embeddim, FC_HIDDEN))).float())
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(FC_HIDDEN, numclasses)

    def attention(self, h):
        v = torch.matmul(self.q, h.transpose(-2, -1)).squeeze(1)
        v = F.softmax(v, -1)
        v_temp = torch.matmul(v.unsqueeze(1), h).transpose(-2, -1)
        return torch.matmul(self.w_h.transpose(1, 0), v_temp).squeeze(2)

    def forward(self, inp_ids, att_mask, token_ids):
        outputs = self.bert(
            input_ids=inp_ids,
            attention_mask=att_mask,
            token_type_ids=token_ids,
        )
        hidden_states = outputs.hidden_states

        cls_stack = torch.stack(
            [hidden_states[layer_i][:, 0].squeeze() for layer_i in range(0, self.numlayers)],
            dim=-1,
        )
        cls_stack = cls_stack.view(-1, self.numlayers, self.embeddim)
        out = self.attention(cls_stack)
        out = self.dropout(out)
        return self.fc(out)


class Bert_LSTM_Attention(nn.Module):
    def __init__(self, numclasses, lstm_hidden=128, dropout=0.1):
        super().__init__()
        self.bert = BertModel.from_pretrained(
            'bert-base-uncased',
            output_hidden_states=False,
            output_attentions=False,
        )
        self.lstm = nn.LSTM(
            input_size=BERT_HIDDEN,
            hidden_size=lstm_hidden,
            batch_first=True,
            bidirectional=True,
        )
        self.attn = nn.Linear(lstm_hidden * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(lstm_hidden * 2, numclasses)

    def forward(self, inp_ids, att_mask, token_ids):
        outputs = self.bert(
            input_ids=inp_ids,
            attention_mask=att_mask,
            token_type_ids=token_ids,
        )
        sequence_output = outputs.last_hidden_state
        lstm_out, _ = self.lstm(sequence_output)

        # Masked token-attention so padding tokens do not influence pooling.
        attn_scores = self.attn(lstm_out).squeeze(-1)
        attn_scores = attn_scores.masked_fill(att_mask == 0, torch.finfo(attn_scores.dtype).min)
        attn_weights = F.softmax(attn_scores, dim=-1).unsqueeze(1)

        context = torch.bmm(attn_weights, lstm_out).squeeze(1)
        return self.fc(self.dropout(context))

## 4. Training and Evaluation

In [8]:
def evaluate(loader, net, device, model_name):
    net.eval()

    with torch.no_grad():
        total_loss = 0.0
        total = 0.0
        correct = 0.0
        y_pred, y_true = [], []

        for input_id, attention_masks, token_ids, labels in loader:
            input_id = input_id.to(device)
            attention_masks = attention_masks.to(device)
            token_ids = token_ids.to(device)
            labels = labels.long().to(device)

            if model_name == 'base':
                loss, output = net(input_id, attention_masks, token_ids, labels)
            else:
                output = net(input_id, attention_masks, token_ids)
                loss = F.cross_entropy(output, labels, reduction='sum')

            total_loss += loss.item()
            preds = torch.argmax(output, 1)
            y_pred.extend(preds.tolist())
            y_true.extend(labels.tolist())
            correct += torch.sum(preds == labels).item()
            total += input_id.size(0)

        macro_f1 = round(f1_score(y_true, y_pred, average='macro'), 2) * 100
        return round(total_loss / total, 3), round((correct / total) * 100, 2), macro_f1

In [9]:
def build_model(model_name, numclasses, device):
    if model_name == 'lstm':
        return Bert_LSTM(numclasses).to(device)
    if model_name == 'attention':
        return Bert_Attention(numclasses).to(device)
    if model_name == 'lstm_attention':
        return Bert_LSTM_Attention(numclasses).to(device)
    return Bert_Base(numclasses).to(device)


def maybe_freeze_bert_layers(model, freeze_bert=False, freeze_n_layers=0):
    if not freeze_bert:
        return

    bert_backbone = None
    if hasattr(model, 'bert') and hasattr(model.bert, 'encoder'):
        bert_backbone = model.bert
    elif hasattr(model, 'bert') and hasattr(model.bert, 'bert'):
        bert_backbone = model.bert.bert

    if bert_backbone is None:
        return

    for p in bert_backbone.embeddings.parameters():
        p.requires_grad = False

    for layer_idx, layer in enumerate(bert_backbone.encoder.layer):
        if layer_idx < freeze_n_layers:
            for p in layer.parameters():
                p.requires_grad = False


def train_model_kfold(
    input_ids,
    attention_masks,
    token_type_ids,
    labels,
    test_loader,
    model_name,
    numclasses,
    numepochs,
    k_folds,
    batch_size,
    device,
    max_len,
    save_prefix='kfold',
    early_stopping_patience=2,
    use_amp=True,
    freeze_bert=False,
    freeze_n_layers=0,
    skip_existing_folds=True,
):
    labels_np = labels.cpu().numpy()
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=SEED)

    fold_results = []
    best_overall_acc = -1.0
    best_overall_state_dict = None
    amp_enabled = use_amp and device.type == 'cuda'

    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(labels_np)), labels_np), start=1):
        print(f"\n{'=' * 72}")
        print(f"TRAINING {model_name.upper()} - FOLD {fold_idx}/{k_folds}")
        print(f"{'=' * 72}")

        fold_save_path = f"{save_prefix}_{model_name}_fold{fold_idx}.pth"

        if skip_existing_folds and os.path.exists(fold_save_path):
            checkpoint = torch.load(fold_save_path, map_location=device)
            model = build_model(model_name, numclasses, device)
            model.load_state_dict(checkpoint['model_state_dict'])
            _, test_acc, test_f1 = evaluate(test_loader, model, device, model_name)
            fold_results.append({'fold': fold_idx, 'test_acc': test_acc, 'test_f1': test_f1})
            if test_acc > best_overall_acc:
                best_overall_acc = test_acc
                best_overall_state_dict = copy.deepcopy(model.state_dict())
            continue

        train_idx_t = torch.tensor(train_idx, dtype=torch.long)
        val_idx_t = torch.tensor(val_idx, dtype=torch.long)

        fold_train_loader = get_loader(
            input_ids[train_idx_t],
            attention_masks[train_idx_t],
            token_type_ids[train_idx_t],
            labels[train_idx_t],
            batch_size,
            shuffle=True,
        )
        fold_val_loader = get_loader(
            input_ids[val_idx_t],
            attention_masks[val_idx_t],
            token_type_ids[val_idx_t],
            labels[val_idx_t],
            batch_size,
            shuffle=False,
        )

        model = build_model(model_name, numclasses, device)
        maybe_freeze_bert_layers(model, freeze_bert=freeze_bert, freeze_n_layers=freeze_n_layers)

        optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-5)
        scaler = torch.amp.GradScaler(device.type, enabled=amp_enabled)

        best_val_acc = -1.0
        best_fold_state_dict = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0

        for epoch in range(1, numepochs + 1):
            model.train()
            for input_id, att_masks, token_ids, batch_labels in fold_train_loader:
                input_id = input_id.to(device)
                att_masks = att_masks.to(device)
                token_ids = token_ids.to(device)
                batch_labels = batch_labels.long().to(device)

                model.zero_grad()

                with torch.amp.autocast(device_type=device.type, enabled=amp_enabled):
                    if model_name == 'base':
                        loss, _ = model(input_id, att_masks, token_ids, batch_labels)
                    else:
                        output = model(input_id, att_masks, token_ids)
                        loss = F.cross_entropy(output, batch_labels)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

            val_loss, val_acc, val_f1 = evaluate(fold_val_loader, model, device, model_name)
            print(f"Fold {fold_idx} Epoch {epoch} | Val Loss {val_loss} | Val Acc {val_acc} | Val F1 {val_f1}")

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_fold_state_dict = copy.deepcopy(model.state_dict())
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1

            if epochs_without_improvement >= early_stopping_patience:
                break

        model.load_state_dict(best_fold_state_dict)
        _, test_acc, test_f1 = evaluate(test_loader, model, device, model_name)
        fold_results.append({'fold': fold_idx, 'test_acc': test_acc, 'test_f1': test_f1})

        torch.save(
            {
                'model_state_dict': best_fold_state_dict,
                'model_type': model_name,
                'numclasses': numclasses,
                'maxlen': max_len,
                'fold': fold_idx,
                'k_folds': k_folds,
            },
            fold_save_path,
        )

        if test_acc > best_overall_acc:
            best_overall_acc = test_acc
            best_overall_state_dict = copy.deepcopy(best_fold_state_dict)

    best_model = build_model(model_name, numclasses, device)
    best_model.load_state_dict(best_overall_state_dict)
    best_model.eval()

    best_model_save_path = f"{save_prefix}_best_{model_name}_model.pth"
    torch.save(
        {
            'model_state_dict': best_overall_state_dict,
            'model_type': model_name,
            'numclasses': numclasses,
            'maxlen': max_len,
            'k_folds': k_folds,
        },
        best_model_save_path,
    )

    avg_test_acc = float(np.mean([x['test_acc'] for x in fold_results]))
    avg_test_f1 = float(np.mean([x['test_f1'] for x in fold_results]))

    print(f"Average {model_name.upper()} Test Accuracy: {avg_test_acc:.2f}")
    print(f"Average {model_name.upper()} Test F1: {avg_test_f1:.2f}")

    return best_model, fold_results, best_model_save_path

## 5. Run K-Fold Training

In [10]:
dataset = 'laptop'   # 'laptop' or 'restaurant'
numclasses = 3
maxlen = 80
data_path = '../Data/'

batch_size = 8
numepochs = 4
k_folds = 5

early_stopping_patience = 2
use_amp = True
freeze_bert = True
freeze_n_layers = 8
skip_existing_folds = True

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [11]:
train_sentence, train_aspect, train_sentiment, test_sentence, test_aspect, test_sentiment = load_semeval(
    dataset=dataset,
    num_classes=numclasses,
    data_path=data_path,
)

print(f'Train samples: {len(train_sentence)}')
print(f'Test samples: {len(test_sentence)}')

----------------------------------------
Max train sentence length: 78
Max train aspect length: 6
----------------------------------------
Train samples: 2313
Test samples: 638


In [12]:
bert_tokenizer = get_pretrained_tokenizer()
print('Tokenizer loaded.')

Tokenizer loaded.


In [13]:
train_input_ids, train_attention_masks, train_token_type_ids = tokenize_sentences(
    bert_tokenizer,
    train_sentence,
    train_aspect,
    maxlen,
)
test_input_ids, test_attention_masks, test_token_type_ids = tokenize_sentences(
    bert_tokenizer,
    test_sentence,
    test_aspect,
    maxlen,
)

train_labels = torch.tensor(train_sentiment, dtype=torch.long)
test_labels = torch.tensor(test_sentiment, dtype=torch.long)

print('Tokenization completed.')

Tokenization completed.


In [14]:
train_loader = get_loader(
    train_input_ids,
    train_attention_masks,
    train_token_type_ids,
    train_labels,
    batch_size,
    shuffle=True,
)

test_loader = get_loader(
    test_input_ids,
    test_attention_masks,
    test_token_type_ids,
    test_labels,
    batch_size,
    shuffle=False,
)

print(f'Train batches: {len(train_loader)}')
print(f'Test batches: {len(test_loader)}')

Train batches: 290
Test batches: 80


In [ ]:
model_names = ['lstm', 'attention', 'base', 'lstm_attention']
trained_models = {}
model_results = {}
best_checkpoint_paths = {}
all_fold_results = {}

for model_name in model_names:
    best_model, fold_results, checkpoint_path = train_model_kfold(
        input_ids=train_input_ids,
        attention_masks=train_attention_masks,
        token_type_ids=train_token_type_ids,
        labels=train_labels,
        test_loader=test_loader,
        model_name=model_name,
        numclasses=numclasses,
        numepochs=numepochs,
        k_folds=k_folds,
        batch_size=batch_size,
        device=DEVICE,
        max_len=maxlen,
        save_prefix='kfold',
        early_stopping_patience=early_stopping_patience,
        use_amp=use_amp,
        freeze_bert=freeze_bert,
        freeze_n_layers=freeze_n_layers,
        skip_existing_folds=skip_existing_folds,
    )

    trained_models[model_name] = best_model
    best_checkpoint_paths[model_name] = checkpoint_path
    all_fold_results[model_name] = fold_results

    train_loss, train_acc, train_f1 = evaluate(train_loader, best_model, DEVICE, model_name)
    test_loss, test_acc, test_f1 = evaluate(test_loader, best_model, DEVICE, model_name)
    kfold_avg_test_acc = round(float(np.mean([x['test_acc'] for x in fold_results])), 2)
    kfold_avg_test_f1 = round(float(np.mean([x['test_f1'] for x in fold_results])), 2)

    model_results[model_name] = {
        'train_loss': train_loss,
        'train_acc': train_acc,
        'train_f1': train_f1,
        'test_loss': test_loss,
        'test_acc': test_acc,
        'test_f1': test_f1,
        'kfold_avg_test_acc': kfold_avg_test_acc,
        'kfold_avg_test_f1': kfold_avg_test_f1,
    }

print('\n' + '=' * 114)
print('MODEL COMPARISON SUMMARY')
print('=' * 114)
print(f"{'Model':<16} {'Train Loss':<12} {'Train Acc':<11} {'Train F1':<10} {'Test Loss':<11} {'Test Acc':<10} {'Test F1':<9} {'KF Avg Acc':<11} {'KF Avg F1':<10}")
print('-' * 114)
for name, result in model_results.items():
    print(
        f"{name.upper():<16} {result['train_loss']:<12} {result['train_acc']:<11} {result['train_f1']:<10} "
        f"{result['test_loss']:<11} {result['test_acc']:<10} {result['test_f1']:<9} "
        f"{result['kfold_avg_test_acc']:<11} {result['kfold_avg_test_f1']:<10}"
    )


TRAINING LSTM - FOLD 1/5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9244.90it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



TRAINING LSTM - FOLD 2/5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10428.25it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



TRAINING LSTM - FOLD 3/5


...


TRAINING LSTM_ATTENTION - FOLD 4/5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9820.76it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



TRAINING LSTM_ATTENTION - FOLD 5/5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9197.73it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10267.26it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
----------------------------

Average LSTM_ATTENTION Test Accuracy: 76.30
Average LSTM_ATTENTION Test F1: 70.40

MODEL COMPARISON SUMMARY
Model            Train Acc    Test Acc     KF Avg Acc   KF Avg F1   
------------------------------------------------------------------------------------
LSTM             86.86        77.74        75.8         69.0        
ATTENTION        86.6         77.43        76.65        70.8        
BASE             87.98        76.96        76.24        71.2        
LSTM_ATTENTION   86.21        78.21        76.3         70.4        


In [16]:
best_model_name = max(model_results, key=lambda x: model_results[x]['kfold_avg_test_acc'])
print(f"\nBest model: {best_model_name.upper()} ({model_results[best_model_name]['kfold_avg_test_acc']}%)")

print('\nSaved checkpoints:')
for name, path in best_checkpoint_paths.items():
    print(f'  - {name.upper()}: {path}')


Best model: ATTENTION (76.65%)

Saved checkpoints:
  - LSTM: kfold_best_lstm_model.pth
  - ATTENTION: kfold_best_attention_model.pth
  - BASE: kfold_best_base_model.pth
  - LSTM_ATTENTION: kfold_best_lstm_attention_model.pth


## 5.1 Detailed Evaluation Metrics

In [ ]:
print('\nPER-FOLD TEST METRICS')
print('=' * 64)
for name in model_names:
    print(f"\n{name.upper()}")
    print('-' * 64)
    print(f"{'Fold':<8} {'Test Acc':<12} {'Test F1':<12}")
    for row in all_fold_results[name]:
        print(f"{row['fold']:<8} {row['test_acc']:<12} {row['test_f1']:<12}")

print('\nAGGREGATED METRICS')
print('=' * 64)
print(f"{'Model':<16} {'Train Loss':<12} {'Train Acc':<10} {'Train F1':<10} {'Test Loss':<11} {'Test Acc':<10} {'Test F1':<10}")
for name, r in model_results.items():
    print(
        f"{name.upper():<16} {r['train_loss']:<12} {r['train_acc']:<10} {r['train_f1']:<10} "
        f"{r['test_loss']:<11} {r['test_acc']:<10} {r['test_f1']:<10}"
    )

## 6. Inference

In [17]:
def load_model(model_path, device):
    checkpoint = torch.load(model_path, map_location=device)
    model_type = checkpoint['model_type']
    num_classes = checkpoint['numclasses']
    max_len = checkpoint['maxlen']

    model = build_model(model_type, num_classes, device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    return model, max_len, model_type


def predict_sentiment(model, tokenizer, sentence, aspect, max_len, device, model_type):
    encoded = tokenizer(
        text=sentence,
        text_pair=aspect,
        add_special_tokens=True,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        return_tensors='pt',
    )

    input_ids = encoded['input_ids'].to(device)
    attention_mask = encoded['attention_mask'].to(device)
    token_type_ids = encoded['token_type_ids'].to(device)

    with torch.no_grad():
        if model_type == 'base':
            dummy_label = torch.tensor([0]).to(device)
            _, output = model(input_ids, attention_mask, token_type_ids, dummy_label)
        else:
            output = model(input_ids, attention_mask, token_type_ids)

        probabilities = F.softmax(output, dim=1)
        confidence, predicted = torch.max(probabilities, 1)

    return predicted.item(), confidence.item()


label_names = {0: 'Negative', 1: 'Positive', 2: 'Neutral'}

example_sentence = 'The battery life is amazing but the keyboard is average.'
example_aspect = 'battery life'

model_path = best_checkpoint_paths[best_model_name]
infer_model, infer_maxlen, infer_type = load_model(model_path, DEVICE)
pred_label, confidence = predict_sentiment(
    infer_model,
    bert_tokenizer,
    example_sentence,
    example_aspect,
    infer_maxlen,
    DEVICE,
    infer_type,
)

print(f'Model: {best_model_name.upper()}')
print(f'Sentence: {example_sentence}')
print(f'Aspect: {example_aspect}')
print(f"Prediction: {label_names[pred_label]} (confidence: {confidence:.2%})")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7239.34it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model: ATTENTION
Sentence: The battery life is amazing but the keyboard is average.
Aspect: battery life
Prediction: Positive (confidence: 94.84%)


## 7. Optional QLoRA (BERT Base)

In [ ]:
def run_qlora_single(train_sentence, train_aspect, train_sentiment, test_sentence, test_aspect, test_sentiment,
                     model_name='bert-base-uncased', output_dir='qlora_bert_base_run', max_length=80,
                     num_train_epochs=3, learning_rate=2e-4, train_batch_size=8, eval_batch_size=16,
                     lora_r=16, lora_alpha=32, lora_dropout=0.05):
    try:
        from datasets import Dataset
        from transformers import (
            AutoTokenizer,
            AutoModelForSequenceClassification,
            Trainer,
            TrainingArguments,
            DataCollatorWithPadding,
            BitsAndBytesConfig,
        )
        from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
    except Exception as e:
        print('QLoRA dependencies missing:', repr(e))
        print('Install: pip install -U transformers peft bitsandbytes accelerate datasets')
        return None

    if not torch.cuda.is_available():
        print('QLoRA requires CUDA for 4-bit bitsandbytes path.')
        return None

    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

    def build_ds(sentences, aspects, labels):
        ds = Dataset.from_dict({'text': sentences, 'text_pair': aspects, 'labels': labels})
        return ds.map(
            lambda x: tokenizer(x['text'], x['text_pair'], truncation=True, max_length=max_length),
            batched=True,
            remove_columns=['text', 'text_pair'],
        )

    train_ds = build_ds(train_sentence, train_aspect, train_sentiment)
    test_ds = build_ds(test_sentence, test_aspect, test_sentiment)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=numclasses,
        quantization_config=bnb_config,
        device_map='auto',
    )
    model = prepare_model_for_kbit_training(model)
    model.config.use_cache = False

    lora_cfg = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        bias='none',
        target_modules=['query', 'key', 'value'],
        task_type=TaskType.SEQ_CLS,
    )
    model = get_peft_model(model, lora_cfg)

    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=num_train_epochs,
        learning_rate=learning_rate,
        per_device_train_batch_size=train_batch_size,
        per_device_eval_batch_size=eval_batch_size,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,
        fp16=True,
        report_to='none',
        logging_steps=20,
        load_best_model_at_end=True,
        metric_for_best_model='eval_macro_f1',
        greater_is_better=True,
    )

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {
            'accuracy': float((preds == labels).mean()),
            'macro_f1': float(f1_score(labels, preds, average='macro')),
        }

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        tokenizer=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
    )

    trainer.train()
    eval_metrics = trainer.evaluate(test_ds)

    trainer.model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    print('\nQLoRA Evaluation Metrics')
    print(eval_metrics)
    return eval_metrics

In [ ]:
enable_qlora = False

if enable_qlora:
    qlora_metrics = run_qlora_single(
        train_sentence=train_sentence,
        train_aspect=train_aspect,
        train_sentiment=train_sentiment,
        test_sentence=test_sentence,
        test_aspect=test_aspect,
        test_sentiment=test_sentiment,
        max_length=maxlen,
        output_dir='kfold_bert_base_qlora_single',
    )
else:
    print('QLoRA is disabled. Set enable_qlora=True to run it.')